# GastroLib Demonstration

This notebook walks through the primary GastroLib modules

In [ ]:
# Import the public APIs used in this walkthrough
from gastrolib import (
    aggregate_recipe_nutrition_from_ingredients,
    build_ingredient_cooccurrence_network,
    build_cuisine_ingredient_network,
    compute_cuisine_similarity,
    compute_pairing_matrix,
    compute_recipe_size_distribution,
    plot_cuisine_similarity_heatmap,
    load_sample_cuisine_dataset,
    mine_frequent_itemsets,
    plot_category_composition_pie,
    plot_ingredient_network,
    plot_pairing_heatmap,
    summarize_ingredients,
    plot_frequency_rank_comparison,
    compare_cuisine_randomizations
)

import matplotlib.pyplot as plt

In [ ]:
# Load the sample dataset bundled for the demo
dataset, nutrition_table = load_sample_cuisine_dataset()
recipes = dataset.to_dataframe()
recipes.head()

In [ ]:
# Start with simple structural statistics
size_result = compute_recipe_size_distribution(recipes)
print("Recipe size statistics:")
for key, value in size_result["stats"].items():
    print(f"  {key}: {value}")

# Then inspect ingredient usage across cuisines
ingredient_stats = summarize_ingredients(recipes, cuisine_col="cuisine")
ingredient_stats.head()

In [ ]:
from gastrolib import compute_ingredient_popularity
ingredient_stats = compute_ingredient_popularity(recipes, top_k=10)
print("Top ingredients by popularity:")
for ing, data in ingredient_stats["global_popularity"].head(10).iterrows():
    print(f"  {data['ingredient']}: {data['count']} recipes")

In [ ]:
from gastrolib import analyze_category_composition
composition = analyze_category_composition(recipes, group_by="cuisine", category_col="category")
print(composition)

In [ ]:
plot_category_composition_pie(composition["composition_table"])

In [ ]:
# Mine common ingredient combinations
itemsets = mine_frequent_itemsets(recipes, min_support=0.4)
itemsets

In [ ]:
pairing_matrix = compute_pairing_matrix(recipes)
pairing_matrix.head()

In [ ]:
plot_pairing_heatmap(pairing_matrix.values, ingredients=list(pairing_matrix.index))

In [ ]:
G = build_ingredient_cooccurrence_network(recipes)
plot_ingredient_network(G, top_n=30)

In [ ]:
# Compare cuisine-specific ingredient networks
print("Available cuisines:", recipes.cuisine.unique())

# Build Italian ingredient network
italian_network = build_cuisine_ingredient_network(dataset, "italian")
print(f"Italian network: {len(italian_network.nodes)} ingredients, {len(italian_network.edges)} connections")

# Build Indian ingredient network
indian_network = build_cuisine_ingredient_network(dataset, "indian")
print(f"Indian network: {len(indian_network.nodes)} ingredients, {len(indian_network.edges)} connections")

# Inspect the strongest Italian ingredient connections
if len(italian_network.edges) > 0:
    italian_edges = sorted(italian_network.edges(data=True), key=lambda x: x[2].get("weight", 1), reverse=True)[:5]
    print("Top Italian ingredient connections:")
    for u, v, data in italian_edges:
        print(f"  {u} ↔ {v}: {data.get('weight', 1)} recipes")

In [ ]:
similarity = compute_cuisine_similarity(recipes, min_ingredient_frequency=1)
print(similarity['similarity_matrix'], similarity['cuisines'] )

plot_cuisine_similarity_heatmap(similarity)

In [ ]:
# Aggregate nutrition at the recipe level
recipe_nutrition = aggregate_recipe_nutrition_from_ingredients(
    recipes,
    nutrition_table,
    ingredient_col="ingredients",
    recipe_id_col="recipe_id",
)
recipe_nutrition

# Working with Larger Recipe Datasets

This section demonstrates loading and analyzing a larger, more comprehensive
recipe dataset with multiple cuisines and more examples.


In [ ]:
from gastrolib import RecipeDataset

# Set this to your own local dataset file before running this section
larger_dataset_path = "path/to/your/larger_recipes.csv"

dataset = RecipeDataset.from_csv(larger_dataset_path)
recipes = dataset.to_dataframe()
size_result = compute_recipe_size_distribution(recipes)

print("Recipe size statistics:")
for key, value in size_result["stats"].items():
    print(f"  {key}: {value}")

ingredient_stats = summarize_ingredients(recipes, cuisine_col="cuisine")
ingredient_stats.head()


In [ ]:
from gastrolib import compute_ingredient_popularity
ingredient_stats = compute_ingredient_popularity(recipes, top_k=10)
print("Top ingredients by popularity:")
for ing, data in ingredient_stats["global_popularity"].head(10).iterrows():
    print(f"  {data['ingredient']}: {data['count']} recipes")


In [ ]:
from gastrolib import analyze_category_composition
composition = analyze_category_composition(recipes, group_by="cuisine", category_col="category")
print(composition)

In [ ]:
plot_category_composition_pie(composition["composition_table"])

In [ ]:
print("Available cuisines:", recipes.cuisine.unique())

# Build Italian ingredient network
italian_network = build_cuisine_ingredient_network(dataset, "italian")
print(f"Italian network: {len(italian_network.nodes)} ingredients, {len(italian_network.edges)} connections")

# Build Indian ingredient network
indian_network = build_cuisine_ingredient_network(dataset, "indian")
print(f"Indian network: {len(indian_network.nodes)} ingredients, {len(indian_network.edges)} connections")

# Strongest connections in Italian cuisine
if len(italian_network.edges) > 0:
    italian_edges = sorted(italian_network.edges(data=True), key=lambda x: x[2].get("weight", 1), reverse=True)[:5]
    print("Top Italian ingredient connections:")
    for u, v, data in italian_edges:
        print(f"  {u} ↔ {v}: {data.get('weight', 1)} recipes")

In [ ]:
similarity = compute_cuisine_similarity(recipes, min_ingredient_frequency=1)
print(similarity['similarity_matrix'], similarity['cuisines'] )

plot_cuisine_similarity_heatmap(similarity)

In [ ]:
G = build_ingredient_cooccurrence_network(recipes)
plot_ingredient_network(G, top_n=50, top_label_n=50)



This section demonstrates how to create randomized versions of cuisines
to understand what makes real cuisines unique through comparison.

We implement four types of randomization:
1. **Random**: Completely random ingredients
2. **Frequency Preserved**: Popular ingredients stay popular
3. **Category Preserved**: Ingredient types are preserved
4. **Frequency + Category Preserved**: Most realistic randomization

In [ ]:
# Compare real cuisines against randomized baselines
cuisines_list = ["Mexican", "Indian", "Italian"]

for sample_cuisine in cuisines_list:
    comparisons = compare_cuisine_randomizations(recipes, sample_cuisine)
    print(sample_cuisine + " cuisine randomization analysis:")
    for method, data in comparisons.items():
        print(f"  {method}: {len(data)} recipes")

    # Visualize how ingredient ranks shift under randomization
    plt.figure(figsize=(12, 8))
    plot_frequency_rank_comparison(comparisons)
    plt.title(sample_cuisine + ": Real vs Randomized Ingredient Patterns", fontsize=14)
    plt.show()